# Prompt

1. sourch: https://www.youtube.com/watch?v=DV4e2n-dIv0&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv&index=8


In [1]:
# get the documents that are coming depending on search engine 

import requests
from minsearch import Index


def search(question, course="llm-zoomcamp", num_results=5):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=num_results
    )


######################################################
# 1. Get your data set, documents (a list of dictionaries)
#######################################################
# get the available courses
# each doc has course, course_path, path, questions_count fields
# the path contians the post past, "https://datatalks.club/faq" + "/json/data-engineering-zoomcamp.json" is wher ethe doc answer/question is saved 
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()



documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    # get a list of dictionaries from a specific course
    course_data = course_response.json()
    # extend the list
    documents.extend(course_data)




######################################################
# 2. Get your data set, documents (a list of dictionaries)
# Note:: we will fit the misearch with the dataset that we have 
# then given the question, we get the context from the the search result
#######################################################
index = Index(
    text_fields=["question", "section", "answer"], # fields that conatin useful information to get the answer (fields that are used to perfom search)
    keyword_fields=["course"], # this is used when I want to filter, for example select * from documents where section="Machine Learning for Classification" then I will ignore all the courses and I will use only that section,
)
# fit with the documents (json file that contains all the prepared data)
index.fit(documents)

# get search result
question = "I just discovered the course. Can I join now?"
search_results = search(question, course="llm-zoomcamp", num_results=5)
search_results



[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [ ]:
######################################################
# 3. Building the context (rearrange the search result)
# Each document becomes a block with the section, question, and answer. This format makes it easy for the LLM to read. We turned a list of dictionaries into one string. It's a small preprocessing step before we send the data to the LLM
#######################################################
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

# context = build_context(search_results)

In [4]:
search_results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [3]:
search_results[0]["section"]

'General Course-Related Questions'

In [7]:
lines = []

for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")
        break

In [8]:
lines

['General Course-Related Questions',
 'Q: I just discovered the course. Can I still join?',
 'A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 '']

In [ ]:





######################################################################################################################
# 3. Building the prompt (Given the templte, put the search result and the question together to give it to llm
######################################################################################################################
# instead of this, I will build a prompt function
# from langchain_core.prompts import ChatPromptTemplate
# prompt = ChatPromptTemplate.from_messages([
#     (
#         "system", 
#         "Your task is to answer questions from the course participants
#          based on the provided context.
#          Use the context to find relevant information and provide accurate
#          answers. If the answer is not found in the context,
#          respond with "I don't know.
#     ),
#     ("human", "{question}")
# ])



INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

def build_prompt(question, search_results):
    # rearrage the search_results so that llm model can read it easily (I think this is for each model, we need to build one)
    context = build_context(search_results)
    # the prompt needs to have question and answers
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

# try it
prompt = build_prompt(question, search_results)
print(prompt)

In [6]:
"\n".join(lines).strip()

'General Course-Related Questions\nQ: I just discovered the course. Can I still join?\nA: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nGeneral Course-Related Questions\nQ: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nA: You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\nGeneral Course-Related Questions\nQ: Certificate: Can I follow the course in a self-paced mode and get a certificate?\nA: No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time

In [11]:
USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )

'\nQuestion:\nI just discovered the course. Can I join now?\n\nContext:\nGeneral Course-Related Questions\nQ: I just discovered the course. Can I still join?\nA: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nGeneral Course-Related Questions\nQ: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nA: You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\nGeneral Course-Related Questions\nQ: Certificate: Can I follow the course in a self-paced mode and get a certificate?\nA: No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after s

In [ ]:


llm = ChatOllama(
    model="gemma3:4b",
    temperature=0.9,
    base_url="http://localhost:11434",
)

chain = prompt | llm | StrOutputParser()
chain.invoke({  "context": docs,
     "question":  "I just discovered the course. Can I join now?"
})